# Universal Atari agent eval / replay

Watch replays and check returns for any trained Atari agent (dqn, rainbow, mdqn, dreamer).

**Only edit the cell below**: pick the config and the checkpoint. Everything else auto-adapts to the agent family.

In [ ]:
from coop_rl.configs.dreamer_atari import get_config

CHECKPOINT = "/home/sia/coop-rl_results/run-0u1dynz8/chkpt_train_step_0076000"

In [ ]:
import importlib
import itertools

import jax
import jax.numpy as jnp
import numpy as np

from coop_rl.base.environment import HandlerEnvDreamerAtari

conf = get_config()

# Auto-detect the agent family from the configured env handler.
is_dreamer = conf.env is HandlerEnvDreamerAtari

# The single-observation select_action lives in the same agent module the collector's
# (batched) select_action comes from. Resolving it generically makes this notebook work
# for dqn / rainbow / mdqn / dreamer with no edits.
agent_mod = importlib.import_module(conf.args_collector.get_select_action_fn.__module__)

if is_dreamer:
    obs_space, _, act_space = conf.env.check_env(
        env_name=conf.args_env.env_name,
        num_envs=1,
        screen_size=conf.args_env.screen_size,
        sticky=conf.args_env.sticky,
    )
    conf.observation_shape = obs_space
    conf.actions_shape = act_space
else:
    conf.observation_shape, conf.observation_dtype, conf.actions_shape = conf.env.check_env(
        conf.args_env.env_name, conf.args_env.stack_size
    )

conf.args_state_recover.checkpointdir = CHECKPOINT
conf.args_state_recover.rng = jax.random.PRNGKey(0)
flax_state = conf.state_recover(**conf.args_state_recover)

if is_dreamer:
    # Stateful: select_action threads flax_state (RSSM carry) and takes an obs dict.
    select_action = agent_mod.get_select_action_fn(flax_state)
else:
    # Stateless: the network was trained on float32 obs normalised to [0, 1]. This
    # preprocessing must match the collector's, or the Q-values are garbage.
    select_action = agent_mod.get_select_action_fn(
        flax_state.apply_fn,
        obs_preprocess_fn=lambda x: x.astype(jnp.float32) / 255.0,
    )

print(f"Agent family: {'dreamer' if is_dreamer else 'dqn-family'} ({agent_mod.__name__})")
print(f"Env: {conf.args_env.env_name}")
print(f"Checkpoint: {CHECKPOINT}")

In [ ]:
import ale_py
import gymnasium as gym
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation

from coop_rl.base.environment import FirstDimToLast

ale_py.ALEInterface.setLoggerMode(ale_py.LoggerMode.Error)

if is_dreamer:
    # Single grayscale-frame env (exactly what the agent sees); rendered directly.
    env = conf.env(
        env_name=conf.args_env.env_name,
        num_envs=1,
        screen_size=conf.args_env.screen_size,
        sticky=conf.args_env.sticky,
    )
else:
    # HandlerEnvAtari is a vector env and cannot render, so build the env directly with
    # the same preprocessing stack used during training, plus rgb_array rendering.
    env = gym.make(
        conf.args_env.env_name,
        frameskip=1,
        repeat_action_probability=0,
        render_mode="rgb_array",
    )
    env = AtariPreprocessing(env, terminal_on_life_loss=False, grayscale_obs=True, scale_obs=False)
    if conf.args_env.stack_size > 1:
        env = FrameStackObservation(env, conf.args_env.stack_size)
        env = FirstDimToLast(env)

In [ ]:
_rng = jax.random.PRNGKey(0)


def rollout_episode(seed=None, render=True):
    """Run one episode; return (frames, total_reward).

    Threads the global agent state across calls: ``flax_state`` (Dreamer RSSM carry)
    or ``_rng`` (DQN-family action sampling), mirroring how the collectors run.
    For Dreamer, frames are the 96x96 grayscale agent observation; for the DQN family
    they are full-resolution RGB from ``env.render()``.
    """
    global flax_state, _rng
    frames = []
    total_reward = 0.0

    if is_dreamer:
        img, _ = env.reset(seed=seed)
        # is_first=True resets the RSSM carry inside the policy (as the collector relies on).
        obs = {
            "image": img,
            "is_first": np.ones(1, bool),
            "is_last": np.zeros(1, bool),
            "is_terminal": np.zeros(1, bool),
            "reward": np.zeros(1, np.float32),
        }
        for _ in itertools.count(start=1):
            if render:
                frames.append(obs["image"][0, :, :, 0].copy())
            flax_state, acts, _ = select_action(flax_state, obs)
            act = np.asarray(acts["action"], dtype=np.int32)
            next_img, reward, terminated, truncated, _ = env.step(act)
            done = np.logical_or(terminated, truncated)
            total_reward += float(reward.sum())
            obs = {
                "image": next_img,
                "is_first": np.zeros(1, bool),
                "is_last": done,
                "is_terminal": done,
                "reward": reward.astype(np.float32),
            }
            if done.any():
                break
    else:
        observation, _ = env.reset(seed=seed)
        for _ in itertools.count(start=1):
            if render:
                frames.append(env.render())
            _rng, action_jnp = select_action(_rng, flax_state.params, observation)
            observation, reward, terminated, truncated, _ = env.step(np.asarray(action_jnp).item())
            total_reward += reward
            if terminated or truncated:
                break

    return frames, total_reward

## Single-episode replay

In [ ]:
from IPython.display import HTML, display
from matplotlib import animation
from matplotlib import pyplot as plt

frames, total_reward = rollout_episode(seed=None, render=True)
print(f"Total steps: {len(frames)}")
print(f"Total reward: {total_reward:.1f}")

# Render collected frames as an inline animation.
fig, ax = plt.subplots(figsize=(4, 4))
ax.axis("off")
if frames[0].ndim == 2:
    im = ax.imshow(frames[0], cmap="gray", vmin=0, vmax=255)
else:
    im = ax.imshow(frames[0])
anim = animation.FuncAnimation(
    fig, lambda f: im.set_data(f), frames=frames, interval=33, blit=False
)
plt.close(fig)
display(HTML(anim.to_jshtml()))

## Multi-episode evaluation

A single episode is high variance; average over N to get a return comparable to what the collectors report.

In [ ]:
N = 10
returns = [rollout_episode(seed=s, render=False)[1] for s in range(N)]
for i, r in enumerate(returns):
    print(f"Episode {i + 1:2d}: {r:.1f}")
print(
    f"\nMean over {N} episodes: {np.mean(returns):.2f}  "
    f"(std {np.std(returns):.2f}, min {np.min(returns):.0f}, max {np.max(returns):.0f})"
)

In [ ]:
env.close()